In [4]:
from dotenv import load_dotenv

load_dotenv('../.env')

import anthropic

client = anthropic.Anthropic()

In [6]:
import json
from pathlib import Path

PASTA_SKILLS = Path("skills")
PASTA_RULES = Path("rules")


def parse_skill_md(caminho: Path) -> tuple[dict, str]:
    """
    Lê um arquivo SKILL.md e separa o frontmatter (YAML entre "---") do corpo.

    Retorna:
        metadata: dict com os campos do frontmatter (ex: name, description)
        corpo: texto das instruções da skill, sem o frontmatter
    """
    texto = caminho.read_text(encoding="utf-8")
    _, frontmatter_bruto, corpo = texto.split("---", 2)

    metadata = {}
    for linha in frontmatter_bruto.strip().splitlines():
        chave, _, valor = linha.partition(":")
        metadata[chave.strip()] = valor.strip()

    return metadata, corpo.strip()


def carregar_skills(pasta: Path) -> dict:
    """
    Descobre todas as skills disponíveis (uma pasta por skill, cada uma com um SKILL.md)
    e indexa pelo campo "name" do frontmatter.

    Só o "name" e a "description" de cada skill ficam expostos por padrão
    (ver tool_skill abaixo) — é isso que entra na janela de contexto o tempo todo.
    O "corpo" (instruções completas) só é lido para dentro do contexto quando
    a skill é efetivamente carregada, via tool call.
    """
    skills = {}
    for skill_md in sorted(pasta.glob("*/SKILL.md")):
        metadata, corpo = parse_skill_md(skill_md)
        skills[metadata["name"]] = {
            "description": metadata["description"],
            "corpo": corpo,
        }
    return skills


def carregar_rules(pasta: Path) -> str:
    """
    Carrega o conteúdo de todos os arquivos .md de uma pasta de rules
    e os concatena em um único bloco de texto, para ser injetado no
    system prompt do agente.
    """
    blocos = [arquivo.read_text(encoding="utf-8") for arquivo in sorted(pasta.glob("*.md"))]
    return "\n\n".join(blocos)


skills_disponiveis = carregar_skills(PASTA_SKILLS)

# Assim como o Claude Code faz na prática: cada skill vira uma linha "nome: description"
# dentro da própria descrição da tool "Skill". Isso é o que fica sempre visível para o
# modelo decidir SE e QUAL skill invocar — leve, e sem carregar o corpo de nenhuma delas.
tool_skill = {
    "name": "Skill",
    "description": (
        "Carrega as instruções completas de uma skill disponível, pelo nome. "
        "Invoque quando a mensagem do usuário se encaixar na descrição de uma das skills abaixo:\n\n"
        + "\n".join(f"- {nome}: {info['description']}" for nome, info in skills_disponiveis.items())
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "nome_skill": {
                "type": "string",
                "enum": list(skills_disponiveis.keys()),
                "description": "Nome (campo 'name' do frontmatter) da skill a ser carregada",
            }
        },
        "required": ["nome_skill"],
    },
}


def processar_tool_call(tool_name: str, tool_input: dict) -> str:
    """Processa a chamada de uma tool e retorna o resultado"""
    if tool_name == "Skill":
        nome_skill = tool_input.get("nome_skill")
        skill = skills_disponiveis.get(nome_skill)
        if skill is None:
            return f"Skill '{nome_skill}' não encontrada."
        # É só aqui que o corpo da skill (as instruções completas) entra na janela de
        # contexto — como resultado da tool call, e não antes disso.
        return skill["corpo"]
    return f"Tool '{tool_name}' não reconhecida"


def executar_tool_calls(response) -> list:
    """Extrai e executa todas as tool calls da resposta"""
    resultados = []

    for bloco in response.content:
        if bloco.type == "tool_use":
            resultado = processar_tool_call(bloco.name, bloco.input)

            resultados.append({
                "tool_use_id": bloco.id,
                "nome_tool": bloco.name,
                "resultado": resultado,
            })

    return resultados


# Rules (comportamento sempre ativo) ficam concatenadas no final do system prompt,
# do mesmo jeito que em 005_rules — diferente das skills, que só entram no contexto
# quando a tool "Skill" é chamada.
system_prompt = (
    "Responda da forma mais breve possivel, sem rodeios, e de forma objetiva. "
    "Evite respostas longas e detalhadas.\n\n"
    "Siga rigorosamente as regras abaixo:\n\n"
    f"{carregar_rules(PASTA_RULES)}"
)
tools = [tool_skill]
historico = []
rodada = 0

print("Chat iniciado. Digite 'sair' para encerrar.\n")

while True:
    entrada_usuario = input("Você: ")

    if entrada_usuario.lower() == "sair":
        break

    historico.append({"role": "user", "content": entrada_usuario})

    # Loop interno: repete enquanto o Claude solicitar tool calls (ex: carregar uma skill)
    while True:
        rodada += 1

        enviado = {
            "system": system_prompt,
            "tools": tools,
            "messages": [dict(msg) for msg in historico],
        }

        print(f"\n{f'====== Rodada {rodada} — ENVIADO ======':=<50}")
        print(json.dumps(enviado, indent=4, ensure_ascii=False))

        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=1024,
            system=system_prompt,
            messages=historico,
            tools=tools,
        )

        recebido = {
            "stop_reason": response.stop_reason,
            "content": [bloco.model_dump() for bloco in response.content],
        }

        print(f"\n{f'------ Rodada {rodada} — RECEBIDO ------':-<50}")
        print(json.dumps(recebido, indent=4, ensure_ascii=False))
        print(''.ljust(50, '='))

        historico.append({"role": "assistant", "content": recebido["content"]})

        if response.stop_reason != "tool_use":
            break

        # Chamada -> execução -> conclusão das tool calls.
        # É neste ponto que o corpo da skill (se invocada) passa a existir dentro do
        # "messages" reenviado à API — mimetizando o "progressive disclosure" real.
        resultados = executar_tool_calls(response)

        historico.append({
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": r["tool_use_id"],
                    "content": r["resultado"],
                }
                for r in resultados
            ]
        })


Chat iniciado. Digite 'sair' para encerrar.


====== Rodada 1 — ENVIADO ========================
{
    "system": "Responda da forma mais breve possivel, sem rodeios, e de forma objetiva. Evite respostas longas e detalhadas.\n\nSiga rigorosamente as regras abaixo:\n\n# Regra: Elogio Inicial na Resposta\n\n## Contexto\n\nEsta regra define um comportamento obrigatório de estilo para as respostas do assistente, aplicado via `system prompt`.\n\n## Regra\n\n**Toda resposta deve começar com um elogio de exatamente 5 palavras, relacionado ao conteúdo da pergunta do usuário.**\n\nApós o elogio, a resposta deve continuar normalmente, respondendo à pergunta do usuário.\n\n### Exemplos\n\n| Pergunta do usuário | Início da resposta |\n|---|---|\n| \"Como funciona um loop for em Python?\" | \"Ótima pergunta sobre estruturas de repetição! Um loop for...\" |\n| \"Qual a capital da França?\" | \"Curiosidade geográfica muito interessante e válida! A capital...\" |\n| \"Como criar uma função em JavaScrip